In [2]:
import bilby 
from bilby.core.prior import Uniform
from bilby.gw.utils import asd_from_freq_series

import numpy as np
import matplotlib
matplotlib.use("Qt5Agg") 
import matplotlib.pyplot as plt
%matplotlib widget
import h5py
import lalsimulation as lalsim
import lal
import os

from scipy.interpolate import UnivariateSpline
from scipy.optimize import root_scalar

import papermill as pm
import scrapbook as sb

/home/selmavangstein/miniconda3/envs/ringdown/lib/python3.11/site-packages/lalsimulation/lalsimulation.py:8: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


In [3]:
outdir = "outdir"
filepath = 'chombo/GRChombo_BBSsol02_A17A17q100d17p000_Res40.h5' 
#filepath = 'chombo/GRChombo_BBSsol02_A147A147q100d12p000_Res40.h5'

luminosity_distance = 175. #MPc
mtotal = 71.5 # in solar masses

beginning_taper_safety = 2
end_taper_safety = 8
pad_factor = 4  # Pad with signal_length on each side

inclination = 0.78
phiRef = 0.0 #reference phase
psi = 1.329 #polarization angle
ra = 2.333 #right ascension
dec = 0.190 #declination

bandpass_filter = True
window_filter = False
taper_end = True
dense_star = True
plotting = True

In [4]:
#some inverse fourier transform helper 
def infft(frequency_domain_strain, sampling_frequency, length=None):
    """ Inverse FFT for use in conjunction with nfft.

    Parameters
    ----------
    frequency_domain_strain: array_like
        Single-sided, normalised FFT of the time-domain strain data (in units
        of strain / Hz).
    sampling_frequency: int, float
        Sampling frequency of the data.
    length: float
        length of the transformed axis of the output.
    """

    time_domain_strain_norm = np.fft.irfft(frequency_domain_strain, n=length)
    time_domain_strain = time_domain_strain_norm * sampling_frequency
    return time_domain_strain

In [5]:
# Specify the output directory and the name of the simulation.
label = "phenomXP"
bilby.core.utils.setup_logger(outdir=outdir, label=label)
f = h5py.File(filepath, 'r')

In [6]:
# setting up dict to feed parameters into waveforms
params = lal.CreateDict()
lalsim.SimInspiralWaveformParamsInsertNumRelData(params, filepath)

inc = 0 #this needs to be 0 for tapering calc - will adjust later

# Metadata parameters masses
# extract masses and convert to different units
m1 = f.attrs['mass1'] #code units
m2 = f.attrs['mass2']

mass_1 = m1 * mtotal / (m1 + m2) #solar masses
mass_2 = m2 * mtotal / (m1 + m2)

# Choose extrinsic parameters

m1SI = mass_1 * lal.MSUN_SI #in kg
m2SI = mass_2 * lal.MSUN_SI

spins = lalsim.SimInspiralNRWaveformGetSpinsFromHDF5File(0., mtotal, filepath)
s1x, s1y, s1z = spins[0], spins[1], spins[2]
s2x, s2y, s2z = spins[3], spins[4], spins[5]

# Set sampling frequency of the data segment that we're going to inject the signal into
# just be aware of aliasing issues if too low
sampling_frequency = 4096.0  # Hz
deltaT = 1.0/sampling_frequency #cadence
distance = luminosity_distance * lal.PC_SI * 1.0e6


# we need to set the lowest trustable frequency - set as lowest simulated frequency, scaled by the chosen mass. Will change after tapering
f_lower = f.attrs['f_lower_at_1MSUN']/mtotal  # this choice generates the whole NR waveforms from the beginning
fRef = 0   #beginning of the waveform
fStart = f_lower


In [7]:
f.close()

In [8]:
params = lal.CreateDict()
lalsim.SimInspiralWaveformParamsInsertNumRelData(params, filepath)

approx = lalsim.NR_hdf5

inject_l_modes=[2]
ModeArray = lalsim.SimInspiralCreateModeArray()
for mode in inject_l_modes:
    lalsim.SimInspiralModeArrayActivateAllModesAtL(ModeArray, mode)

lalsim.SimInspiralWaveformParamsInsertModeArray(params, ModeArray)

h_p, h_c = lalsim.SimInspiralChooseTDWaveform(m1SI, m2SI, s1x, s1y, s1z,
                s2x, s2y, s2z, distance, inc, phiRef, np.pi/2., 0.0, 0.0, 
                deltaT, fStart, fRef, params, approx)

times = np.arange(len(h_p.data.data))*h_p.deltaT

In [9]:
# get stuff for tapering calculation

phase = np.arctan2(h_c.data.data, h_p.data.data)
unwrapped_phase = np.unwrap(phase)
unwrapped_phase_interp = UnivariateSpline(times, unwrapped_phase, k=3, s=0)
omega_interp = unwrapped_phase_interp.derivative() 
frequency = omega_interp(times) / (2.0 * np.pi)

safety = beginning_taper_safety
sol = root_scalar(lambda t1: (1+safety)*(1./t1)-omega_interp(t1)/(2.0*np.pi),
                    x0 = 0.1, 
                    bracket=[0.01, 0.3])
t0, t1 = 0., sol.root
f0, f1 = omega_interp(t0)/(2.0*np.pi), omega_interp(t1)/(2.0*np.pi)
print(f"{t0=:.2e}, {f0=:.2e}, {t1=:.2e}, {f1=:.2e}")

minimum_frequency = f1


t0=0.00e+00, f0=2.08e+01, t1=1.28e-01, f1=2.34e+01


In [10]:
inc = inclination

h_p, h_c = lalsim.SimInspiralChooseTDWaveform(m1SI, m2SI, s1x, s1y, s1z,
                s2x, s2y, s2z, distance, inc, phiRef, np.pi/2., 0.0, 0.0, 
                deltaT, fStart, fRef, params, approx)

times = np.arange(len(h_p.data.data))*h_p.deltaT

In [11]:
def window(times, t0, t1, direction='on'):
    taper = np.ones_like(times)
    mask = (times >= t0) & (times <= t1)
    x = (times[mask] - t0) / (t1 - t0)
    taper[mask] = 0.5 * (1 - np.cos(np.pi * x))

    if direction == 'off':
        # Flip the taper and set regions outside [t0, t1] appropriately
        taper[mask] = 1 - taper[mask]
        taper[times < t0] = 1.0
        taper[times > t1] = 0.0
    else:  # direction == 'on'
        taper[times < t0] = 0.0
        taper[times > t1] = 1.0
        
    return taper

In [12]:
wf = 'cosine'

taper = window(times, t0, t1, direction='on')
tapered_hp = h_p.data.data * taper
tapered_hc = h_c.data.data * taper

In [13]:
def find_peaks(hp):
    """
    Find peaks by sign change in derivative: dhp/dt goes + -> -.
    Returns array of indices (index corresponds to sample where sign change occurs).
    """
    d_hp = np.diff(hp)
    # look for positive -> negative sign change
    sign_change = (d_hp[:-1] > 0) & (d_hp[1:] < 0)
    peaks = np.where(sign_change)[0] + 1
    return peaks

def nth_last_peak(hp, times, n=1):
    peaks = find_peaks(hp)
    nth_idx = peaks[-n]
    nth_time = times[nth_idx]
    nth_value = hp[nth_idx]
    return nth_idx, nth_time, nth_value

n = end_taper_safety
idx_taper = find_peaks(h_p.data.data)[-n]

In [14]:
t_start = times[idx_taper] if idx_taper < len(times) else times[-1]
t_end = times[-1]

if t_start >= t_end:
    raise ValueError(f"Invalid taper interval: t_start ({t_start}) >= t_end ({t_end})")

end_taper = window(times, t_start, t_end, direction='off')
# combine with the existing start taper
taper = taper * end_taper

# update the stored tapers and tapered signals
tapered_hp = tapered_hp * end_taper
tapered_hc = tapered_hc * end_taper

In [ ]:
signal_length = len(times)
pad_length = int(signal_length * pad_factor)

# Get the time step (assuming uniform sampling)
dt = times[1] - times[0]
pad_time = pad_length * dt

# Create new time array with padding
times_before = times[0] - np.arange(pad_length, 0, -1) * dt
times_after = times[-1] + np.arange(1, pad_length + 1) * dt
# Combine all times
times_padded = np.concatenate([times_before, times, times_after])
# shift to start at t=0
times_padded = times_padded-times_padded[0]

# Pad with zeros on each side
padded_hp = np.pad(tapered_hp, pad_length, mode='constant', constant_values=0)
padded_hc = np.pad(tapered_hc, pad_length, mode='constant', constant_values=0)
padded_taper = np.pad(taper, pad_length, mode='constant', constant_values=0)

print(f"Original signal length: {signal_length} points ({times[0]:.4f} s to {times[-1]:.4f} s)")
print(f"Padded signal length: {len(times_padded)} points ({times_padded[0]:.4f} s to {times_padded[-1]:.4f} s)")
print(f"Padding: {pad_length} points ({pad_length * dt:.4f} s) on each side")
print(f"Signal now occupies {100 * signal_length / len(times_padded):.1f}% of total length")

Original signal length: 2482 points (0.0000 s to 0.6057 s)
Padded signal length: 22338 points (0.0000 s to 5.4534 s)
Padding: 9928 points (2.4238 s) on each side
Signal now occupies 11.1% of total length


In [16]:
# apply band pass filter before injecting into ifos
from scipy.signal import butter, sosfiltfilt
def bandpass_filter(data, lowcut, highcut, fs, order=4):
    sos = butter(order, [lowcut, highcut], btype='band', fs=fs, output='sos')
    y = sosfiltfilt(sos, data)
    return y
lowcut = minimum_frequency  # Hz
highcut = 270  # Hz - by eye
fs = 4096.0  # Sampling frequency in Hz
filtered_hp = bandpass_filter(padded_hp, lowcut, highcut, fs)
filtered_hc = bandpass_filter(padded_hc, lowcut, highcut, fs)

In [17]:
preprocessed_hp = filtered_hp if (bandpass_filter or window_filter) else padded_hp 
preprocessed_hc = filtered_hc if (bandpass_filter or window_filter) else padded_hc

In [18]:
def create_nr_injection(times_array, hp_array, hc_array):
    """
    Factory function that creates a custom NR injection function
    with specific time series data.
    """
    def nr_injection(time):
            
        hp = np.interp(time, times_array, hp_array)
        hc = np.interp(time, times_array, hc_array)
        return {"plus": hp, "cross": hc}
    
    return nr_injection

In [19]:
# propagate and inject each tapered signal
# create dict to store the data
original_hp = np.copy(h_p.data.data)
original_hc = np.copy(h_c.data.data)

amplitude = np.sqrt(preprocessed_hp**2 + preprocessed_hc**2)
peak_id = np.argmax(amplitude)
peak = times_padded[peak_id]

hplus = preprocessed_hp[peak_id]
hcross = preprocessed_hc[peak_id]
phase_merger = np.arctan2(-hcross,hplus) + np.pi


#duration = times_padded[-1]
start_time = times_padded[0]
signal_start_time = times_padded[pad_length]
duration = len(times_padded) / sampling_frequency

np.random.seed(88170235)
injection_parameters = dict(
    mass_1=mass_1,
    mass_2=mass_2,
    a_1=0.0,
    a_2=0.0,
    tilt_1=0.0,
    tilt_2=0.0,
    phi_12=0.0,
    phi_jl=0.0,
    luminosity_distance=luminosity_distance,
    theta_jn=inc,
    psi=psi, #used to be np.pi/2
    phase=phase_merger,
    geocent_time=0,
    ra=ra,
    dec=dec
)

# bibly assumes a signal centered at 0 and then applies a geocentric time shift, so we must shift our signal accordingly
# roll the arrays so the peak is at index 0
#hp_rolled = np.roll(preprocessed_hp, -peak_id)
#hc_rolled = np.roll(preprocessed_hc, -peak_id)
hp_rolled = preprocessed_hp
hc_rolled = preprocessed_hc

nr_injection_wf = create_nr_injection(times_padded, hp_rolled, hc_rolled)

# propagate to interferometers
waveform = bilby.gw.waveform_generator.WaveformGenerator(duration=duration, sampling_frequency=sampling_frequency,
        time_domain_source_model=nr_injection_wf, start_time=0);

# get time and frequency domain strains and arrays
time_domain = waveform.time_domain_strain(parameters=injection_parameters);
time_array = waveform.time_array

fr_domain = waveform.frequency_domain_strain(parameters=injection_parameters);
fr_array = waveform.frequency_array

11:51 bilby INFO    : Waveform generator instantiated: WaveformGenerator(duration=5.45361328125, sampling_frequency=4096.0, start_time=0, frequency_domain_source_model=None, time_domain_source_model=__main__.nr_injection, parameter_conversion=bilby.gw.conversion.convert_to_lal_binary_black_hole_parameters, waveform_arguments={})


In [20]:
# create ifos and inject signal and noise
ifos = bilby.gw.detector.InterferometerList(['H1', 'L1']);
ifos.set_strain_data_from_power_spectral_densities(
    sampling_frequency=sampling_frequency, duration=duration,
    start_time=start_time);
ifos.inject_signal(waveform_generator=waveform,
                parameters=injection_parameters, raise_error=False);

11:51 bilby INFO    : Injected signal in H1:
11:51 bilby INFO    :   optimal SNR = 80.76
11:51 bilby INFO    :   matched filter SNR = 80.04-2.36j
11:51 bilby INFO    :   mass_1 = 35.75
11:51 bilby INFO    :   mass_2 = 35.75
11:51 bilby INFO    :   a_1 = 0.0
11:51 bilby INFO    :   a_2 = 0.0
11:51 bilby INFO    :   tilt_1 = 0.0
11:51 bilby INFO    :   optimal SNR = 80.76
11:51 bilby INFO    :   matched filter SNR = 80.04-2.36j
11:51 bilby INFO    :   mass_1 = 35.75
11:51 bilby INFO    :   mass_2 = 35.75
11:51 bilby INFO    :   a_1 = 0.0
11:51 bilby INFO    :   a_2 = 0.0
11:51 bilby INFO    :   tilt_1 = 0.0
11:51 bilby INFO    :   tilt_2 = 0.0
11:51 bilby INFO    :   phi_12 = 0.0
11:51 bilby INFO    :   phi_jl = 0.0
11:51 bilby INFO    :   luminosity_distance = 175.0
11:51 bilby INFO    :   theta_jn = 0.78
11:51 bilby INFO    :   psi = 1.329
11:51 bilby INFO    :   tilt_2 = 0.0
11:51 bilby INFO    :   phi_12 = 0.0
11:51 bilby INFO    :   phi_jl = 0.0
11:51 bilby INFO    :   luminosity_di

In [21]:
#locate the merger (peak amplitude) in the ifo data
ifo = ifos[0]  # pick one interferometer to plot
times_ifo = ifo.time_array
strain = ifo.time_domain_strain
amplitude = np.sqrt(strain**2)
peak_id_ifo = np.argmax(amplitude)
peak_time_ifo = times_ifo[peak_id_ifo]

## Time Shifting and Data Export

Now we shift the time arrays so that:
1. The reference geocenter time is set to match GW150914 paper values (1420878141.235932 GPS)
2. Detector arrival times are calculated accounting for light travel time from geocenter
3. Peak amplitudes in each detector align with their respective arrival times
4. All saved data uses GPS time coordinates consistent with the ringdown analysis script

In [ ]:
# === TIME SHIFTING AND DATA SAVING ===
# Set reference geocenter time (using the paper's value for consistency)
reference_geocent_time = 1420878141.235932  # GPS time from GW150914 paper Table II

# Calculate arrival times at each detector
h1_delay = ifos[0].time_delay_from_geocenter(ra, dec, reference_geocent_time)
l1_delay = ifos[1].time_delay_from_geocenter(ra, dec, reference_geocent_time)

h1_arrival_time = reference_geocent_time + h1_delay
l1_arrival_time = reference_geocent_time + l1_delay

print(f"\n=== TIME CONFIGURATION ===")
print(f"Reference geocenter time: {reference_geocent_time:.6f} GPS")
print(f"H1 arrival time: {h1_arrival_time:.6f} GPS")
print(f"L1 arrival time: {l1_arrival_time:.6f} GPS")
print(f"H1 delay from geocenter: {h1_delay*1000:.3f} ms")
print(f"L1 delay from geocenter: {l1_delay*1000:.3f} ms")

# Find peak in each detector's strain
h1_strain = ifos[0].time_domain_strain
l1_strain = ifos[1].time_domain_strain
h1_time_array = ifos[0].time_array
l1_time_array = ifos[1].time_array

h1_amplitude = np.abs(h1_strain)
l1_amplitude = np.abs(l1_strain)
h1_peak_idx = np.argmax(h1_amplitude)
l1_peak_idx = np.argmax(l1_amplitude)

h1_peak_time_relative = h1_time_array[h1_peak_idx]
l1_peak_time_relative = l1_time_array[l1_peak_idx]

print(f"\nPeak locations in original time array:")
print(f"H1 peak at: {h1_peak_time_relative:.6f} s (relative)")
print(f"L1 peak at: {l1_peak_time_relative:.6f} s (relative)")

# Calculate shifts needed to align peaks with arrival times
# The shifted time array should have: peak_time_shifted = arrival_time
# So: time_original + shift = arrival_time at peak index
# Therefore: shift = arrival_time - peak_time_original
h1_time_shift = h1_arrival_time - h1_peak_time_relative
l1_time_shift = l1_arrival_time - l1_peak_time_relative

print(f"\nTime shifts applied:")
print(f"H1 shift: {h1_time_shift:.6f} s")
print(f"L1 shift: {l1_time_shift:.6f} s")

# Apply time shifts - create shifted time arrays
h1_time_shifted = h1_time_array + h1_time_shift
l1_time_shifted = l1_time_array + l1_time_shift

print(f"\nPeak locations after shifting:")
print(f"H1 peak at: {h1_time_shifted[h1_peak_idx]:.6f} GPS (should be {h1_arrival_time:.6f})")
print(f"L1 peak at: {l1_time_shifted[l1_peak_idx]:.6f} GPS (should be {l1_arrival_time:.6f})")

# Update injection_parameters with the reference geocenter time
injection_parameters['geocent_time'] = reference_geocent_time

# Save data with shifted time arrays
import pandas as pd

# For ringdown analysis - save with H1 time (since ringdown uses H1 as reference)
df_h1_reference = pd.DataFrame({
    'time': h1_time_shifted,
    'H1': h1_strain,
    'L1': l1_strain
})
df_h1_reference.to_csv('bilby_outputs/injection_data.csv', index=False)
print("\ Saved shifted data to 'bilby_outputs/injection_data.csv' (H1 time reference)")

# Also save the old path for backward compatibility
df_h1_reference.to_csv('ringdown_tests/injection_data.csv', index=False)
print("Saved shifted data to 'ringdown_tests/injection_data.csv' (H1 time reference)")



=== TIME CONFIGURATION ===
Reference geocenter time: 1420878141.235932 GPS
H1 arrival time: 1420878141.219014 GPS
L1 arrival time: 1420878141.216519 GPS
H1 delay from geocenter: -16.918 ms
L1 delay from geocenter: -19.413 ms

Peak locations in original time array:
H1 peak at: 2.968262 s (relative)
L1 peak at: 2.965332 s (relative)

Time shifts applied:
H1 shift: 1420878138.250752 s
L1 shift: 1420878138.251187 s

Peak locations after shifting:
H1 peak at: 1420878141.219014 GPS (should be 1420878141.219014)
L1 peak at: 1420878141.216519 GPS (should be 1420878141.216519)

✓ Saved shifted data to 'bilby_outputs/injection_data.csv' (H1 time reference)
Saved shifted data to 'ringdown_tests/injection_data.csv' (H1 time reference)

✓ Saved shifted data to 'bilby_outputs/injection_data.csv' (H1 time reference)
Saved shifted data to 'ringdown_tests/injection_data.csv' (H1 time reference)


In [23]:
print("\n=== METADATA FOR RINGDOWN FIT ===")
print(f"mass:              {mtotal} Msun")
print(f"luminosity dist:   {injection_parameters['luminosity_distance']} Mpc")
print(f"ra:                {injection_parameters['ra']}")
print(f"dec:               {injection_parameters['dec']}")
print(f"psi:               {injection_parameters['psi']}")
print(f"duration:          {duration}")
print(f"H1 SNR:            {ifo.meta_data['optimal_SNR']:.2f}")
print(f"geocent time:      {reference_geocent_time}")
print(f"h1 arrival time:   {h1_arrival_time}")
print(f"h1 peak time:      {h1_time_shifted[h1_peak_idx]}")
print("=================================")


=== METADATA FOR RINGDOWN FIT ===
mass:              71.5 Msun
luminosity dist:   175.0 Mpc
ra:                2.333
dec:               0.19
psi:               1.329
duration:          5.45361328125
H1 SNR:            80.76
geocent time:      1420878141.235932
h1 arrival time:   1420878141.2190142
h1 peak time:      1420878141.2190142


In [ ]:
meta_dict = {
    "mass": mtotal,
    "luminosity_distance": injection_parameters['luminosity_distance'],
    "ra": injection_parameters['ra'],
    "dec": injection_parameters['dec'],
    "psi": injection_parameters['psi'],
    "duration": duration,
    "f_min": minimum_frequency,
    "f_max": highcut,
    "f_ref": fRef,
    "inclination": inc,
    "geocent_time": reference_geocent_time,
    "h1_arrival_time": h1_arrival_time,
    "l1_arrival_time": l1_arrival_time,
    "h1_peak_time": h1_time_shifted[h1_peak_idx],
    "l1_peak_time": l1_time_shifted[l1_peak_idx],
    "sampling_frequency": sampling_frequency,
    "start_time": h1_time_shifted[0],
    "pad_time": pad_time,
}

df = pd.DataFrame([meta_dict])
df.to_csv('bilby_outputs/injection_metadata.csv', index=False)
print("Saved metadata to 'bilby_outputs/injection_metadata.csv'")

Saved metadata to 'bilby_outputs/injection_metadata.csv'


In [122]:
snr = ifos[0].meta_data['optimal_SNR']
snr

np.float64(80.76413238711994)

## Things needed for TDInf

In [123]:
h1_psd = ifos[0].power_spectral_density_array
h1_freq = ifos[0].frequency_array

l1_psd = ifos[1].power_spectral_density_array
l1_freq = ifos[1].frequency_array

#replace inf with 0.25 in psd
h1_psd[np.isinf(h1_psd)] = 0.25
l1_psd[np.isinf(l1_psd)] = 0.25

# create array of pairs (freq, psd) for each ifo
h1_psd_array = np.column_stack((h1_freq, h1_psd))
l1_psd_array = np.column_stack((l1_freq, l1_psd))

# save each array to .dat file
np.savetxt('H1_PSD.dat', h1_psd_array)
np.savetxt('L1_PSD.dat', l1_psd_array)

In [ ]:
# Initialize results dictionary with all keys set to None, then fill with available values
import math

# All possible keys
_all_keys = [
    'H1_matched_filter_abs_snr','H1_matched_filter_snr_angle','H1_optimal_snr',
    'H1_spcal_amp_0','H1_spcal_amp_1','H1_spcal_amp_2','H1_spcal_amp_3','H1_spcal_amp_4',
    'H1_spcal_amp_5','H1_spcal_amp_6','H1_spcal_amp_7','H1_spcal_amp_8','H1_spcal_amp_9',
    'H1_spcal_phase_0','H1_spcal_phase_1','H1_spcal_phase_2','H1_spcal_phase_3','H1_spcal_phase_4',
    'H1_spcal_phase_5','H1_spcal_phase_6','H1_spcal_phase_7','H1_spcal_phase_8','H1_spcal_phase_9',
    'L1_matched_filter_abs_snr','L1_matched_filter_snr_angle','L1_optimal_snr',
    'L1_spcal_amp_0','L1_spcal_amp_1','L1_spcal_amp_2','L1_spcal_amp_3','L1_spcal_amp_4',
    'L1_spcal_amp_5','L1_spcal_amp_6','L1_spcal_amp_7','L1_spcal_amp_8','L1_spcal_amp_9',
    'L1_spcal_phase_0','L1_spcal_phase_1','L1_spcal_phase_2','L1_spcal_phase_3','L1_spcal_phase_4',
    'L1_spcal_phase_5','L1_spcal_phase_6','L1_spcal_phase_7','L1_spcal_phase_8','L1_spcal_phase_9',
    'V1_matched_filter_abs_snr','V1_matched_filter_snr_angle','V1_optimal_snr',
    'V1_spcal_amp_0','V1_spcal_amp_1','V1_spcal_amp_2','V1_spcal_amp_3','V1_spcal_amp_4',
    'V1_spcal_amp_5','V1_spcal_amp_6','V1_spcal_amp_7','V1_spcal_amp_8','V1_spcal_amp_9',
    'V1_spcal_phase_0','V1_spcal_phase_1','V1_spcal_phase_2','V1_spcal_phase_3','V1_spcal_phase_4',
    'V1_spcal_phase_5','V1_spcal_phase_6','V1_spcal_phase_7','V1_spcal_phase_8','V1_spcal_phase_9',
    'azimuth','cosalpha','cos_theta_jn','deltalogL','deltaloglH1','deltaloglL1','deltaloglV1',
    'log_likelihood','log_prior','logw','network_matched_filter_snr','network_optimal_snr',
    'phase','phi_12','phi_jl','mass_ratio','t0','geocent_time','ra','dec','luminosity_distance',
    'psi','chirp_mass','a_1','a_2','tilt_1','tilt_2','theta_jn','inverted_mass_ratio','mass_1',
    'mass_2','total_mass','symmetric_mass_ratio','iota','spin_1x','spin_1y','spin_1z','spin_2x',
    'spin_2y','spin_2z','phi_1','phi_2','chi_eff','chi_p','final_mass','final_spin','final_kick',
    'cos_tilt_1','cos_tilt_2','redshift','comoving_distance','mass_1_source','mass_2_source',
    'total_mass_source','chirp_mass_source','final_mass_source','radiated_energy','H1_time','L1_time',
    'V1_time','H1_matched_filter_snr','L1_matched_filter_snr','V1_matched_filter_snr','cos_iota',
    'peak_luminosity','f_ref','geocenter_time','right_ascension','declination','polarization',
    'inclination','spin1_x','spin1_y','spin1_z','spin2_x','spin2_y','spin2_z','spin1_magnitude',
    'spin2_magnitude'
]

# Initialize all to None
results_dict = {k: None for k in _all_keys}

# Create extra_data dict for values not in _all_keys
extra_data = {}

# === Now fill in values we have ===

# Extrinsic parameters - use the reference geocenter time
results_dict['geocent_time'] = reference_geocent_time
results_dict['geocenter_time'] = reference_geocent_time
results_dict['t0'] = reference_geocent_time
results_dict['ra'] = injection_parameters['ra']
results_dict['right_ascension'] = injection_parameters['ra']
results_dict['dec'] = injection_parameters['dec']
results_dict['declination'] = injection_parameters['dec']
results_dict['psi'] = injection_parameters['psi']
results_dict['polarization'] = injection_parameters['psi']
results_dict['theta_jn'] = injection_parameters['theta_jn']
results_dict['inclination'] = injection_parameters['theta_jn']
results_dict['iota'] = injection_parameters['theta_jn']
results_dict['luminosity_distance'] = injection_parameters['luminosity_distance']

# Masses
m1 = injection_parameters['mass_1']
m2 = injection_parameters['mass_2']
results_dict['mass_1'] = m1
results_dict['mass_2'] = m2
m_tot = m1 + m2
results_dict['total_mass'] = m_tot
mc = (m1 * m2)**(3.0/5.0) / (m_tot**(1.0/5.0))
results_dict['chirp_mass'] = mc
results_dict['symmetric_mass_ratio'] = (m1 * m2) / (m_tot**2)
results_dict['mass_ratio'] = m2 / m1
results_dict['inverted_mass_ratio'] = m1 / m2

# Angles
results_dict['phase'] = injection_parameters['phase']
results_dict['phi_12'] = injection_parameters['phi_12']
results_dict['phi_jl'] = injection_parameters['phi_jl']
results_dict['cos_theta_jn'] = math.cos(injection_parameters['theta_jn'])
results_dict['cos_iota'] = math.cos(injection_parameters['theta_jn'])

# Spins - we have s1x, s1y, s1z, s2x, s2y, s2z from earlier in the notebook
results_dict['a_1'] = injection_parameters['a_1']
results_dict['a_2'] = injection_parameters['a_2']
results_dict['spin1_magnitude'] = injection_parameters['a_1']
results_dict['spin2_magnitude'] = injection_parameters['a_2']
results_dict['tilt_1'] = injection_parameters['tilt_1']
results_dict['tilt_2'] = injection_parameters['tilt_2']
results_dict['cos_tilt_1'] = math.cos(injection_parameters['tilt_1'])
results_dict['cos_tilt_2'] = math.cos(injection_parameters['tilt_2'])

# Spin components (from the variables s1x, s1y, s1z, s2x, s2y, s2z defined earlier)
results_dict['spin_1x'] = s1x
results_dict['spin_1y'] = s1y
results_dict['spin_1z'] = s1z
results_dict['spin_2x'] = s2x
results_dict['spin_2y'] = s2y
results_dict['spin_2z'] = s2z
results_dict['spin1_x'] = s1x
results_dict['spin1_y'] = s1y
results_dict['spin1_z'] = s1z
results_dict['spin2_x'] = s2x
results_dict['spin2_y'] = s2y
results_dict['spin2_z'] = s2z

# # Compute phi_1 and phi_2 from spin components
# if s1x != 0 or s1y != 0:
#     results_dict['phi_1'] = math.atan2(s1y, s1x)
# else:
#     results_dict['phi_1'] = 0.0
    
# if s2x != 0 or s2y != 0:
#     results_dict['phi_2'] = math.atan2(s2y, s2x)
# else:
#     results_dict['phi_2'] = 0.0

# Effective spin parameters
# chi_eff = (m1*chi1z + m2*chi2z) / (m1 + m2)
# results_dict['chi_eff'] = (m1 * s1z + m2 * s2z) / m_tot

# chi_p = max(a1_perp, (4*q+3)/(3*q+4) * q * a2_perp) where q = m2/m1
# a1_perp = math.sqrt(s1x**2 + s1y**2)
# a2_perp = math.sqrt(s2x**2 + s2y**2)
# q = m2 / m1
# chi_p_term2 = ((4*q + 3) / (3*q + 4)) * q * a2_perp
# results_dict['chi_p'] = max(a1_perp, chi_p_term2)

# Reference frequency
results_dict['f_ref'] = fRef

# Per-IFO optimal SNRs from metadata
results_dict['H1_optimal_snr'] = ifos[0].meta_data['optimal_SNR']
results_dict['L1_optimal_snr'] = ifos[1].meta_data['optimal_SNR']

# # Network optimal SNR (quadrature sum)
# results_dict['network_optimal_snr'] = math.sqrt(
#     results_dict['H1_optimal_snr']**2 + results_dict['L1_optimal_snr']**2
# )

# # For matched filter SNR in an injection (no noise), matched filter SNR = optimal SNR
# # and the angle is 0 (perfect alignment)
# results_dict['H1_matched_filter_abs_snr'] = results_dict['H1_optimal_snr']
# results_dict['L1_matched_filter_abs_snr'] = results_dict['L1_optimal_snr']
# results_dict['H1_matched_filter_snr_angle'] = 0.0
# results_dict['L1_matched_filter_snr_angle'] = 0.0

# Network matched filter SNR
results_dict['network_matched_filter_snr'] = results_dict['network_optimal_snr']

# # Complex matched filter SNR (for compatibility)
# results_dict['H1_matched_filter_snr'] = complex(results_dict['H1_optimal_snr'], 0)
# results_dict['L1_matched_filter_snr'] = complex(results_dict['L1_optimal_snr'], 0)

# Per-IFO arrival times using the reference geocenter time
results_dict['H1_time'] = h1_arrival_time
results_dict['L1_time'] = l1_arrival_time

# # Redshift and cosmological parameters
# # Need to find redshift from luminosity distance (inverse problem)
# try:
#     from astropy.cosmology import Planck15, z_at_value
#     import astropy.units as u
    
#     # z_at_value finds the redshift for a given luminosity distance
#     z = z_at_value(Planck15.luminosity_distance, injection_parameters['luminosity_distance'] * u.Mpc)
#     results_dict['redshift'] = float(z)
#     results_dict['comoving_distance'] = float(Planck15.comoving_distance(z).value)  # Mpc
    
#     # Source frame masses
#     results_dict['mass_1_source'] = m1 / (1 + z)
#     results_dict['mass_2_source'] = m2 / (1 + z)
#     results_dict['total_mass_source'] = m_tot / (1 + z)
#     results_dict['chirp_mass_source'] = mc / (1 + z)
# except ImportError:
#     # Fallback: approximate redshift using Hubble law
#     # z ≈ H0 * d_L / c, where H0 ≈ 67.9 km/s/Mpc
#     H0 = 67.9  # km/s/Mpc
#     c_kms = 299792.458  # km/s
#     z_approx = (H0 * injection_parameters['luminosity_distance']) / c_kms
#     results_dict['redshift'] = z_approx
    
#     # Approximate comoving distance for small z: d_c ≈ d_L / (1+z)
#     results_dict['comoving_distance'] = injection_parameters['luminosity_distance'] / (1 + z_approx)
    
#     # Source frame masses
#     results_dict['mass_1_source'] = m1 / (1 + z_approx)
#     results_dict['mass_2_source'] = m2 / (1 + z_approx)
#     results_dict['total_mass_source'] = m_tot / (1 + z_approx)
#     results_dict['chirp_mass_source'] = mc / (1 + z_approx)

# Store arrays and other data in extra_data (not in _all_keys)
extra_data['H1_psd'] = ifos[0].power_spectral_density_array
extra_data['L1_psd'] = ifos[1].power_spectral_density_array
extra_data['H1_asd'] = np.sqrt(ifos[0].power_spectral_density_array)
extra_data['L1_asd'] = np.sqrt(ifos[1].power_spectral_density_array)
extra_data['H1_frequency_array'] = ifos[0].frequency_array
extra_data['L1_frequency_array'] = ifos[1].frequency_array
extra_data['H1_strain'] = ifos[0].time_domain_strain
extra_data['L1_strain'] = ifos[1].time_domain_strain
extra_data['H1_frequency_domain_strain'] = ifos[0].frequency_domain_strain
extra_data['L1_frequency_domain_strain'] = ifos[1].frequency_domain_strain
extra_data['H1_time_array'] = h1_time_shifted  # Use shifted time arrays
extra_data['L1_time_array'] = l1_time_shifted

# Store waveform information
extra_data['waveform_time_array'] = time_array
extra_data['waveform_frequency_array'] = fr_array
extra_data['waveform_time_domain_plus'] = time_domain['plus']
extra_data['waveform_time_domain_cross'] = time_domain['cross']
extra_data['waveform_frequency_domain_plus'] = fr_domain['plus']
extra_data['waveform_frequency_domain_cross'] = fr_domain['cross']

# Store other computed values
extra_data['minimum_frequency'] = minimum_frequency
extra_data['sampling_frequency'] = sampling_frequency
extra_data['duration'] = duration
extra_data['start_time'] = start_time

# Count filled entries
filled = sum(1 for v in results_dict.values() if v is not None)
print(f"\nInitialized {len(results_dict)} keys total in results_dict.")
print(f"Filled {filled} entries with values. Remaining {len(results_dict) - filled} are None.")
print(f"\nStored {len(extra_data)} additional items in extra_data (arrays and computed values).")
print("\nSample filled scalar entries from results_dict:")
sample_keys = ['mass_1', 'mass_2', 'chirp_mass', 'geocent_time', 'H1_time', 'L1_time',
               'H1_optimal_snr', 'L1_optimal_snr', 'network_matched_filter_snr', 
               'chi_eff', 'chi_p', 'redshift', 'mass_1_source']
for k in sample_keys:
    val = results_dict.get(k)
    if val is not None and not isinstance(val, np.ndarray):
        print(f"  {k}: {val}")


Initialized 148 keys total in results_dict.
Filled 50 entries with values. Remaining 98 are None.

Stored 22 additional items in extra_data (arrays and computed values).

Sample filled scalar entries from results_dict:
  mass_1: 35.75
  mass_2: 35.75
  chirp_mass: 31.12218263783643
  geocent_time: 1420878141.235932
  H1_time: 1420878141.2190142
  L1_time: 1420878141.2165194
  H1_optimal_snr: 80.76413238711994
  L1_optimal_snr: 78.08592645871121


In [37]:
# save the results_dict to a csv
df = pd.DataFrame([results_dict])
df.to_csv('bilby_outputs/reference_parameters.csv', index=False)
print("Saved results_dict to 'bilby_outputs/reference_parameters.csv'")

Saved results_dict to 'bilby_outputs/reference_parameters.csv'


In [127]:
print(results_dict['geocent_time'])

1420878141.235932


In [131]:
#find duration of h1_time_shifted
h1_duration = h1_time_shifted[-1] - h1_time_shifted[0]
print(f"H1 data duration: {h1_duration:.2f} s")

H1 data duration: 5.45 s
